# Error Analysis
Find false positives, false negatives, and misclassifications. Understand which classes confuse the model and why.


In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from collections import defaultdict, Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

from src.models.predictor import predict_shelf

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)
CLASS_NAMES = CFG['classes']

MODEL_PATH   = None   # set this to your best.pt path
TEST_IMG_DIR = Path('../data/splits/images/test')
TEST_LBL_DIR = Path('../data/splits/labels/test')
CONF         = 0.5

image_paths = sorted(TEST_IMG_DIR.glob('*.jpg')) + sorted(TEST_IMG_DIR.glob('*.png'))
print(f'Test images: {len(image_paths)}')

In [ ]:
# ── Run predictions on test set ─────────────────────────────────────────────
all_preds = []
all_gts   = []

def parse_gt(lbl_path):
    counts = defaultdict(int)
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return dict(counts)

for img_path in image_paths[:50]:  # limit for notebook speed
    lbl_path = TEST_LBL_DIR / img_path.with_suffix('.txt').name
    gt = parse_gt(lbl_path)
    result = predict_shelf(str(img_path), MODEL_PATH, CONF, '../config/config.yaml')
    pred_counts = defaultdict(int)
    for det in result['detections']:
        cid = CLASS_NAMES.index(det['category']) if det['category'] in CLASS_NAMES else -1
        if cid >= 0:
            pred_counts[cid] += 1
    all_preds.append(dict(pred_counts))
    all_gts.append(gt)

print('Inference complete.')

In [ ]:
# ── Count errors per class ──────────────────────────────────────────────────
over_counts  = defaultdict(int)   # predicted > ground truth
under_counts = defaultdict(int)   # predicted < ground truth

for pred, gt in zip(all_preds, all_gts):
    all_classes = set(pred) | set(gt)
    for cid in all_classes:
        p = pred.get(cid, 0)
        g = gt.get(cid, 0)
        if p > g:
            over_counts[cid] += p - g
        elif p < g:
            under_counts[cid] += g - p

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
cls_labels = CLASS_NAMES

over_vals  = [over_counts.get(i, 0)  for i in range(len(CLASS_NAMES))]
under_vals = [under_counts.get(i, 0) for i in range(len(CLASS_NAMES))]

ax1.bar(cls_labels, over_vals,  color='tomato')
ax1.set_title('Over-detection (FP) per Class')
ax1.set_xlabel('Class')
plt.setp(ax1.get_xticklabels(), rotation=35, ha='right')

ax2.bar(cls_labels, under_vals, color='steelblue')
ax2.set_title('Under-detection (FN) per Class')
ax2.set_xlabel('Class')
plt.setp(ax2.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Confidence distribution: correct vs incorrect predictions ───────────────
correct_confs = []
wrong_confs   = []

for img_path, gt in zip(image_paths[:50], all_gts):
    result = predict_shelf(str(img_path), MODEL_PATH, 0.0, '../config/config.yaml')
    for det in result['detections']:
        cat = det['category']
        cid = CLASS_NAMES.index(cat) if cat in CLASS_NAMES else -1
        conf = det['confidence']
        if cid >= 0 and gt.get(cid, 0) > 0:
            correct_confs.append(conf)
        else:
            wrong_confs.append(conf)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(correct_confs, bins=30, alpha=0.7, label='Correct detections', color='green')
ax.hist(wrong_confs,   bins=30, alpha=0.7, label='False positives',     color='red')
ax.axvline(CONF, color='black', linestyle='--', label=f'Threshold={CONF}')
ax.set_xlabel('Confidence')
ax.set_title('Confidence Distribution: Correct vs. Incorrect Detections')
ax.legend()
plt.tight_layout()
plt.show()

print('Actionable recommendations:')
print('  - Classes with high FN: collect more training images')
print('  - Classes with high FP: add hard-negative examples')
print('  - Low confidence on correct: improve augmentation diversity')